# Lab: kNN and Logistic Regression on the IoT-Flock Healthcare Security Dataset

**Dataset:** IoT Healthcare Security Dataset / ICU Healthcare Security Dataset created using the IoT-Flock tool.

This lab uses normal and malicious IoT healthcare network traffic. The goal is to build and evaluate **k-Nearest Neighbour (kNN)** and **Logistic Regression (LR)** classifiers for security detection.

---

## Learning outcomes

By the end of this lab, students should be able to:

1. Load and inspect an IoT security dataset.
2. Identify **features**, **target labels**, and possible leakage columns.
3. Apply preprocessing for mixed numeric/categorical network-flow data.
4. Implement **kNN from scratch** for binary classification.
5. Train kNN using scikit-learn and tune `k`.
6. Implement **Logistic Regression from scratch** using sigmoid, loss, gradients, and weights.
7. Compare kNN and LR using accuracy, precision, recall, F1-score, ROC-AUC, and confusion matrix.

---

## Lab rules

- Use scikit-learn for preprocessing and final model comparison.
- For the scratch sections, do **not** use `KNeighborsClassifier` or `LogisticRegression`.
- Keep the dataset split fixed using `random_state=42`.
- Always scale numeric features before distance-based models such as kNN.

## SECTION 1 — Dataset loading

### TASK 1.1 — Download and place the dataset

Download the IoT Healthcare Security Dataset and place the CSV or ZIP file either:

```text
same folder as this notebook
```

or

```text
data/
```

### GUIDELINE

The loader below automatically searches for `.csv` or `.zip` files. If you have several files, set `DATA_PATH` manually.

In [3]:
# Core libraries
import os, glob, zipfile, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, RocCurveDisplay
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

In [4]:
# TASK: Put the downloaded IoT-Flock / IoT Healthcare Security dataset CSV file in the same folder
# as this notebook, or inside a folder called data/.
#
# If the dataset is inside a .zip file, put the zip file in the same folder or data/.

DATA_PATH = None  # Optional: set manually if needed


def find_dataset_file(data_path=None):
    """Find a CSV file in the current folder or data/ folder.
    Also extracts the first CSV from a zip file.
    """
    if data_path is not None and os.path.exists(data_path):
        return data_path

    search_patterns = ["*.csv", "data/*.csv", "*.zip", "data/*.zip"]
    files = []

    for pat in search_patterns:
        files.extend(glob.glob(pat))

    csv_files = [f for f in files if f.lower().endswith(".csv")]

    if csv_files:
        return csv_files[0]

    zip_files = [f for f in files if f.lower().endswith(".zip")]

    if zip_files:
        zpath = zip_files[0]
        extract_dir = "data_extracted"
        os.makedirs(extract_dir, exist_ok=True)

        with zipfile.ZipFile(zpath, "r") as z:
            csv_inside = [
                n for n in z.namelist()
                if n.lower().endswith(".csv")
            ]

            if not csv_inside:
                raise FileNotFoundError(
                    "Zip file found, but no CSV file exists inside it."
                )

            z.extract(csv_inside[0], extract_dir)
            return os.path.join(extract_dir, csv_inside[0])

    raise FileNotFoundError(
        "No CSV/ZIP dataset file found. Put the IoT Healthcare Security "
        "dataset CSV/ZIP in the notebook folder or data/ folder."
    )


# Load both datasets
attack_path = "Attack.csv"
environment_path = "environmentMonitoring.csv"

attack_df = pd.read_csv(attack_path)
environment_df = pd.read_csv(environment_path)

# Combine the datasets
df = pd.concat(
    [attack_df, environment_df],
    ignore_index=True
)

# Display dataset information
print("Using datasets:", attack_path, "+", environment_path)
print("Shape:", df.shape)

print("\nClass distribution:")
print(df["class"].value_counts())

# Display first five rows
df.head()

Using datasets: Attack.csv + environmentMonitoring.csv
Shape: (111884, 52)

Class distribution:
class
Attack                   80126
environmentMonitoring    31758
Name: count, dtype: int64


,frame.time_delta,frame.time_relative,frame.len,ip.src,ip.dst,tcp.srcport,tcp.dstport,tcp.flags,tcp.time_delta,tcp.len,...,mqtt.qos,mqtt.retain,mqtt.topic,mqtt.topic_len,mqtt.ver,mqtt.willmsg_len,ip.proto,ip.ttl,class,label
0,0.000000,0.000000,74,10.16.120.44,10.16.120.72,56808,1883,0x00000002,0.000000,0,...,0.0,0.0,0.0,0.0,0.0,0.0,6,64,Attack,1
1,0.000052,0.000052,74,10.16.120.72,10.16.120.44,1883,56808,0x00000012,0.000052,0,...,0.0,0.0,0.0,0.0,0.0,0.0,6,64,Attack,1
2,0.000008,0.000060,74,10.16.120.44,10.16.120.72,56810,1883,0x00000002,0.000000,0,...,0.0,0.0,0.0,0.0,0.0,0.0,6,64,Attack,1
3,0.000012,0.000072,74,10.16.120.72,10.16.120.44,1883,56810,0x00000012,0.000012,0,...,0.0,0.0,0.0,0.0,0.0,0.0,6,64,Attack,1
4,0.000003,0.000075,74,10.16.120.44,10.16.120.72,56812,1883,0x00000002,0.000000,0,...,0.0,0.0,0.0,0.0,0.0,0.0,6,64,Attack,1


## SECTION 2 — Data understanding

### TASK 2.1 — Inspect the dataset

Print:

- dataset shape
- column names
- data types
- missing values
- duplicated rows
- class distribution

### GUIDELINE

For cybersecurity datasets, target columns are often named something like:

```text
label, Label, class, Class, attack, Attack, traffic_type, category
```

### HINT CODE

```python
print(df.shape)
print(df.dtypes)
print(df.isna().sum().sort_values(ascending=False).head(20))
```

In [5]:
# TASK 2.1: Inspect the dataset
# Write your code below

print(" DataSet Info:")
print(df.info())
print(df.dtypes)

print("\n Dataset Shape ")
print(df.shape)

print("\n Dataset columns ")
print(df.columns  )

print("\n Dataset Missing Values ")
print(df.isnull().sum())

print("\nDuplicated rows:")
print(df.duplicated().sum())

print("\nClass distribution:")
print(df["class"].value_counts())

print("\nClass distribution (%):")
print(df["class"].value_counts(normalize=True) * 100)

 DataSet Info:
<class 'pandas.DataFrame'>
RangeIndex: 111884 entries, 0 to 111883
Data columns (total 52 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   frame.time_delta       111884 non-null  float64
 1   frame.time_relative    111884 non-null  float64
 2   frame.len              111884 non-null  int64  
 3   ip.src                 111884 non-null  str    
 4   ip.dst                 111884 non-null  str    
 5   tcp.srcport            111884 non-null  int64  
 6   tcp.dstport            111884 non-null  int64  
 7   tcp.flags              111884 non-null  str    
 8   tcp.time_delta         111884 non-null  float64
 9   tcp.len                111884 non-null  int64  
 10  tcp.ack                111884 non-null  int64  
 11  tcp.connection.fin     111884 non-null  float64
 12  tcp.connection.rst     111884 non-null  float64
 13  tcp.connection.sack    111884 non-null  float64
 14  tcp.connection.syn     111884 no

In [6]:
print(df.columns.tolist())

['frame.time_delta', 'frame.time_relative', 'frame.len', 'ip.src', 'ip.dst', 'tcp.srcport', 'tcp.dstport', 'tcp.flags', 'tcp.time_delta', 'tcp.len', 'tcp.ack', 'tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.sack', 'tcp.connection.syn', 'tcp.flags.ack', 'tcp.flags.fin', 'tcp.flags.push', 'tcp.flags.reset', 'tcp.flags.syn', 'tcp.flags.urg', 'tcp.hdr_len', 'tcp.payload', 'tcp.pdu.size', 'tcp.window_size_value', 'tcp.checksum', 'mqtt.clientid', 'mqtt.clientid_len', 'mqtt.conack.flags', 'mqtt.conack.val', 'mqtt.conflag.passwd', 'mqtt.conflag.qos', 'mqtt.conflag.reserved', 'mqtt.conflag.retain', 'mqtt.conflag.willflag', 'mqtt.conflags', 'mqtt.dupflag', 'mqtt.hdrflags', 'mqtt.kalive', 'mqtt.len', 'mqtt.msg', 'mqtt.msgtype', 'mqtt.qos', 'mqtt.retain', 'mqtt.topic', 'mqtt.topic_len', 'mqtt.ver', 'mqtt.willmsg_len', 'ip.proto', 'ip.ttl', 'class', 'label']


In [7]:
print("CLASS:")
print(df["class"].value_counts())

print("\nLABEL:")
print(df["label"].value_counts())

print("\nFirst 20 rows:")
print(df[["class", "label"]].head(20))

CLASS:
class
Attack                   80126
environmentMonitoring    31758
Name: count, dtype: int64

LABEL:
label
1    80126
0    31758
Name: count, dtype: int64

First 20 rows:
     class  label
0   Attack      1
1   Attack      1
2   Attack      1
3   Attack      1
4   Attack      1
5   Attack      1
6   Attack      1
7   Attack      1
8   Attack      1
9   Attack      1
10  Attack      1
11  Attack      1
12  Attack      1
13  Attack      1
14  Attack      1
15  Attack      1
16  Attack      1
17  Attack      1
18  Attack      1
19  Attack      1


### TASK 2.2 — Identify the target column

Try to automatically detect the target column. If the auto-detection fails, manually set `TARGET_COL`.

### GUIDELINE

The target should represent the traffic class, for example normal vs malicious.

### HINT CODE

```python
candidate_targets = [c for c in df.columns if c.lower() in ['label','class','target','attack','category','traffic_type']]
print(candidate_targets)
```

In [8]:
# TASK 2.2: Identify target column
candidate_targets = [c for c in df.columns if c.lower() in ['label','class','target','attack','category','traffic_type','traffic type']]
print("Candidate target columns:", candidate_targets)

if candidate_targets:
    TARGET_COL = candidate_targets[0]
else:
    TARGET_COL = 'class'  # TODO: set this manually if needed, e.g. TARGET_COL = 'Label'

# TODO: If candidate_targets is not empty, use the first candidate.
print("Selected target column:", TARGET_COL)


# TODO: Print the class distribution.
print("\nClass distribution:")
print(df["class"].value_counts())

print("\nClass distribution (%):")
print(df["class"].value_counts(normalize=True) * 100)

Candidate target columns: ['class', 'label']
Selected target column: class

Class distribution:
class
Attack                   80126
environmentMonitoring    31758
Name: count, dtype: int64

Class distribution (%):
class
Attack                   71.615244
environmentMonitoring    28.384756
Name: proportion, dtype: float64


## SECTION 3 — Cleaning and feature preparation

### TASK 3.1 — Drop leakage or identifier columns

Network datasets often contain columns that are not useful for generalisation or may cause leakage, such as:

```text
id, timestamp, time, src_ip, dst_ip, source_ip, destination_ip, flow_id
```

### GUIDELINE

Do not blindly drop columns. First inspect them. Drop columns that behave like identifiers or timestamps.

In [9]:
# TASK 3.1: Drop leakage / identifier columns
# Hint: create a list of columns to drop based on names.

# Remove exact duplicates
df = df.drop_duplicates().reset_index(drop=True)

# Columns already identified for removal
drop_cols = [
    "frame.time_delta",
    "frame.time_relative",
    "ip.src",
    "ip.dst",
    "tcp.time_delta",
    "mqtt.clientid",
    "tcp.flags.urg",
    "mqtt.conack.val",
    "mqtt.conflag.passwd",
    "mqtt.conflag.qos",
    "mqtt.conflag.reserved",
    "mqtt.conflag.retain",
    "mqtt.conflag.willflag",
    "mqtt.willmsg_len",
    "ip.proto"
]

# Keep the target separate from the features
df_model = df.drop(columns=drop_cols, errors="ignore")
X = df_model.drop(columns=["class", "label"], errors="ignore")
y = df["class"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (111882, 35)
y shape: (111882,)


### TASK 3.2 — Separate X and y

Create:

```python
X = df_model.drop(columns=[TARGET_COL])
y = df_model[TARGET_COL]
```

Then encode the target labels into numeric values.

### GUIDELINE

If the dataset has more than two classes, convert it to binary if appropriate:

- normal / benign → 0
- malicious / attack → 1

If the dataset is already binary, use label encoding.

In [ ]:
# TASK 3.2: Separate features and target
# TODO: create X and y
# TODO: encode y into numeric labels
# TODO: print label mapping



leakage_columns = [col for col in ("class", "label") if col != TARGET_COL]
X = df_model.drop(columns=[TARGET_COL] + leakage_columns, errors="ignore")
y = df_model[TARGET_COL]

print(" X shape:", X.shape)
print(" y shape:", y.shape)

# Encode y into numeric labels

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(le.classes_)

print("\n Encoded target distribution:")
print(pd.Series(y_encoded).value_counts())


### TASK 3.3 — Build preprocessing pipeline

You need:

- median imputation + standard scaling for numeric columns
- most-frequent imputation + one-hot encoding for categorical columns

### HINT CODE

```python
numeric_features = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['int64','float64']).columns.tolist()
```

In [ ]:
# TASK 3.3: Build preprocessing pipeline
# TODO: identify numeric and categorical columns
# TODO: build ColumnTransformer named preprocess

numeric_features = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['int64','float64']).columns.tolist()

# Import Tools

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


## SECTION 4 — Train/test split

### TASK 4.1 — Create a stratified train/test split

Use:

```python
test_size=0.25
random_state=42
stratify=y
```

In [ ]:
# TASK 4.1: Train/test split
# TODO: create X_train, X_test, y_train, y_test
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Display the shapes of the datasets
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

print("\nTraining class distribution:")
print(pd.Series(y_train).value_counts())

print("\nTesting class distribution:")
print(pd.Series(y_test).value_counts())

In [ ]:
import numpy as np
import pandas as pd

# Make copies so the original split stays unchanged
X_train = X_train.copy()
X_test = X_test.copy()

# Convert categorical values to strings, preserving missing values
for col in categorical_features:
    X_train[col] = X_train[col].map(
        lambda v: str(v) if pd.notna(v) else np.nan
    )
    X_test[col] = X_test[col].map(
        lambda v: str(v) if pd.notna(v) else np.nan
    )

# Fit on training data and transform both sets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Convert sparse output to dense arrays if needed
if hasattr(X_train_processed, "toarray"):
    X_train_processed = X_train_processed.toarray()

if hasattr(X_test_processed, "toarray"):
    X_test_processed = X_test_processed.toarray()

# Keep smaller subsets for scratch kNN
MAX_TRAIN = 3000
MAX_TEST = 1000

X_train_knn = X_train_processed[:MAX_TRAIN]
y_train_knn = np.asarray(y_train)[:MAX_TRAIN]

X_test_knn = X_test_processed[:MAX_TEST]
y_test_knn = np.asarray(y_test)[:MAX_TEST]

print("kNN training data:", X_train_knn.shape)
print("kNN testing data:", X_test_knn.shape)

print("Training classes:", np.unique(y_train_knn, return_counts=True))
print("Testing classes:", np.unique(y_test_knn, return_counts=True))

## SECTION 5 — kNN from scratch

### TASK 5.1 — Transform data for scratch kNN

Use the preprocessing pipeline to transform `X_train` and `X_test` into numeric arrays.

### GUIDELINE

For scratch kNN, use a smaller subset first because kNN can be slow on large datasets.

Use for example:

```python
MAX_TRAIN = 3000
MAX_TEST = 1000
```

In [ ]:
# TASK 5.1: Transform data for scratch kNN
# TODO: fit preprocess on X_train and transform X_train, X_test
# TODO: create smaller subsets for scratch implementation
import numpy as np
# Set smaller dataset sizes for scratch kNN
MAX_TRAIN = 3000
MAX_TEST = 1000

# Fit preprocessing on training data and transform it
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test data using the same preprocessing
X_test_processed = preprocessor.transform(X_test)

# Convert sparse arrays to dense arrays if necessary
if hasattr(X_train_processed, "toarray"):
    X_train_processed = X_train_processed.toarray()

if hasattr(X_test_processed, "toarray"):
    X_test_processed = X_test_processed.toarray()

# Select a smaller subset
X_train_knn = X_train_processed[:MAX_TRAIN]
y_train_knn = np.asarray(y_train)[:MAX_TRAIN]

X_test_knn = X_test_processed[:MAX_TEST]
y_test_knn = np.asarray(y_test)[:MAX_TEST]

# Check the resulting shapes
print("kNN training data:", X_train_knn.shape)
print("kNN testing data:", X_test_knn.shape)
print("Training labels:", y_train_knn.shape)
print("Testing labels:", y_test_knn.shape)

### TASK 5.2 — Implement Euclidean distance

Write a function:

```python
euclidean_distance(a, b)
```

where `a` and `b` are numeric vectors.

### HINT CODE

```python
return np.sqrt(np.sum((a - b) ** 2))
```

In [ ]:
# TASK 5.2: Implement Euclidean distance

def euclidean_distance(a, b):
    # TODO: implement distance
    return np.sqrt(np.sum((a - b) ** 2, ))
    

### TASK 5.3 — Implement scratch kNN prediction

Write a function:

```python
knn_predict_one(X_train_arr, y_train_arr, query, k=5)
```

Steps:

1. Compute distance from query to every training row.
2. Sort by distance.
3. Select the `k` nearest neighbours.
4. Return the majority class.

### HINT CODE

```python
distances = np.sqrt(np.sum((X_train_arr - query) ** 2, axis=1))
nearest_idx = np.argsort(distances)[:k]
```

In [ ]:
# TASK 5.3: Implement scratch kNN prediction
from collections import Counter

def knn_predict_one(X_train_arr, y_train_arr, query, k=5):
    # TODO: compute distances
    distances = np.sqrt(
        np.sum((X_train_arr - query) ** 2, axis=1)
    )
    # TODO: identify nearest neighbours
    nearest_indices = np.argsort(distances)[:k]

    nearest_labels = y_train_arr[nearest_indices]

    class_counts = Counter(nearest_labels)
    # TODO: majority vote
    predicted_class = class_counts.most_common(1)[0][0]
    
    return predicted_class


prediction = knn_predict_one(
    X_train_knn,
    y_train_knn,
    X_test_knn[0],
    k=5
)

print("Predicted class:", prediction)
print("Actual class:", y_test_knn[0])

### TASK 5.4 — Evaluate scratch kNN

Evaluate scratch kNN for:

```text
k = 1, 3, 5, 7
```

Report accuracy, precision, recall, and F1-score.

### GUIDELINE

Use `average='weighted'` for precision, recall, and F1-score.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score
)
from sklearn.metrics.pairwise import euclidean_distances

# Calculate all test-to-training distances at once
distances = euclidean_distances(X_test_knn, X_train_knn)

# Find the 7 nearest neighbours (largest k needed)
nearest_idx = np.argpartition(distances, kth=6, axis=1)[:, :7]

# Sort those 7 neighbours by actual distance
nearest_idx = np.take_along_axis(
    nearest_idx,
    np.argsort(np.take_along_axis(distances, nearest_idx, axis=1), axis=1),
    axis=1
)

# Get their labels
neighbour_labels = y_train_knn[nearest_idx]

results = []

for k in [1, 3, 5, 7]:
    y_pred = []

    for labels in neighbour_labels[:, :k]:
        classes, counts = np.unique(labels, return_counts=True)
        y_pred.append(classes[np.argmax(counts)])

    y_pred = np.array(y_pred)

    results.append({
        "k": k,
        "Accuracy": accuracy_score(y_test_knn, y_pred),
        "Precision": precision_score(
            y_test_knn, y_pred, average="weighted", zero_division=0
        ),
        "Recall": recall_score(
            y_test_knn, y_pred, average="weighted", zero_division=0
        ),
        "F1-score": f1_score(
            y_test_knn, y_pred, average="weighted", zero_division=0
        )
    })

results_df = pd.DataFrame(results)
print(results_df.round(4).to_string(index=False))

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Predictions for the selected k (for example, k=5)
print(classification_report(y_test_knn, y_pred, digits=4))
print(confusion_matrix(y_test_knn, y_pred))

## SECTION 6 — scikit-learn kNN with tuning

### TASK 6.1 — Train a kNN pipeline

Use:

```python
Pipeline([
    ('preprocess', preprocess),
    ('model', KNeighborsClassifier())
])
```

Tune:

- `n_neighbors`: 1, 3, 5, 7, 9, 11, 15
- `weights`: uniform, distance
- `metric`: euclidean, manhattan

In [ ]:
# TASK 6.1: kNN pipeline and grid search
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
# TODO: build pipeline
knn_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', KNeighborsClassifier())
])
# TODO: define param_grid
param_grid = {
    'model__n_neighbors': [1, 3, 5, 7, 9, 11, 15],
    'model__weights': ['uniform', 'distance'],
    'model__metric': ['euclidean', 'manhattan']
}
# TODO: run GridSearchCV
grid_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)
# TODO: print best parameters

print("Best parameters:")
print(grid_search.best_params_)

print("\nBest cross-validation F1-score:")
print(grid_search.best_score_)

### TASK 6.2 — Evaluate tuned kNN

Report:

- accuracy
- precision
- recall
- F1-score
- confusion matrix
- classification report
- ROC-AUC if binary

In [ ]:


# TASK 6.2: Evaluate tuned kNN

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# Get the best model from GridSearchCV
best_knn = grid_search.best_estimator_

# Predict the test set
y_pred = best_knn.predict(X_test)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print("Tuned kNN Results")
print("-----------------")
print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

# ROC-AUC for binary classification
# Use probability for the positive class
y_proba = best_knn.predict_proba(X_test)

# Find the column corresponding to the positive class
positive_class = "environmentMonitoring"
positive_index = list(best_knn.classes_).index(positive_class)

roc_auc = roc_auc_score(
    (y_test == positive_class).astype(int),
    y_proba[:, positive_index]
)

print("ROC-AUC:", round(roc_auc, 4))

## SECTION 7 — Logistic Regression from scratch

### TASK 7.1 — Prepare binary labels

Scratch LR below assumes binary classification.

If your dataset has more than two classes, either:

1. filter two classes, or
2. convert to normal vs malicious.

### GUIDELINE

For binary classification, labels must be 0 and 1.

In [ ]:
# TASK 7.1: Prepare binary labels for scratch LR
train_classes = pd.Index(pd.unique(pd.Series(y_train).dropna()))
if len(train_classes) < 2:
    raise ValueError("Scratch logistic regression requires at least two training classes.")

# When there are multiple attack classes, group them as malicious and the rest as normal.
attack_markers = ("attack", "malicious", "anomaly", "intrusion", "dos", "ddos", "flood", "scan")
malicious_classes = [
    label for label in train_classes
    if any(marker in str(label).lower() for marker in attack_markers)
]
if malicious_classes and len(malicious_classes) < len(train_classes):
    label_mapping = {label: int(label in malicious_classes) for label in train_classes}
elif len(train_classes) == 2:
    label_mapping = {train_classes[0]: 0, train_classes[1]: 1}
else:
    raise ValueError(
        "More than two classes were found, but attack classes could not be identified. "
        "Filter two classes or define which labels represent attacks."
    )

if not set(pd.unique(pd.Series(y_test).dropna())).issubset(label_mapping):
    raise ValueError("The test set contains a class that was not present in the training set.")

y_train_binary = pd.Series(y_train).map(label_mapping).to_numpy(dtype=int)
y_test_binary = pd.Series(y_test).map(label_mapping).to_numpy(dtype=int)
if not set(np.unique(y_train_binary)).issubset({0, 1}) or not set(np.unique(y_test_binary)).issubset({0, 1}):
    raise ValueError("Binary target labels must contain only 0 and 1.")

print("Binary label mapping (0 = normal, 1 = malicious):", label_mapping)
print("Training binary class counts:", pd.Series(y_train_binary).value_counts().sort_index().to_dict())
print("Test binary class counts:", pd.Series(y_test_binary).value_counts().sort_index().to_dict())

### TASK 7.2 — Implement sigmoid and binary cross-entropy

Implement:

```python
sigmoid(z)
binary_cross_entropy(y_true, y_prob)
```

In [ ]:
# TASK 7.2: Implement sigmoid and BCE

def sigmoid(z):
    z = np.asarray(z, dtype=float)
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))


def binary_cross_entropy(y_true, y_prob, eps=1e-12):
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    y_prob = np.asarray(y_prob, dtype=float).reshape(-1)
    if y_true.shape != y_prob.shape:
        raise ValueError("y_true and y_prob must have the same shape.")
    y_prob = np.clip(y_prob, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))

### TASK 7.3 — Train logistic regression from scratch

Use gradient descent.

Model:

```text
z = X @ w + b
y_hat = sigmoid(z)
```

Gradients:

```text
dw = (1/n) * X.T @ (y_hat - y)
db = mean(y_hat - y)
```

Updates:

```text
w = w - lr * dw
b = b - lr * db
```

In [ ]:
# TASK 7.3: Train LR from scratch

def train_logistic_regression_scratch(X_train_arr, y_train_arr, lr=0.1, epochs=300):
    X_train_arr = np.asarray(X_train_arr, dtype=float)
    y_train_arr = np.asarray(y_train_arr, dtype=float).reshape(-1)
    if X_train_arr.ndim != 2 or X_train_arr.shape[0] != y_train_arr.size:
        raise ValueError("X_train_arr must be 2D and have one row per target label.")
    if X_train_arr.shape[0] == 0 or X_train_arr.shape[1] == 0:
        raise ValueError("Training data must contain at least one row and one feature.")
    if not np.isin(y_train_arr, [0, 1]).all():
        raise ValueError("Logistic regression targets must be encoded as 0 and 1.")
    if lr <= 0 or epochs <= 0:
        raise ValueError("lr and epochs must be positive.")

    n_samples, n_features = X_train_arr.shape
    weights = np.zeros(n_features, dtype=float)
    bias = 0.0
    loss_history = []

    for _ in range(epochs):
        probabilities = sigmoid(X_train_arr @ weights + bias)
        loss_history.append(binary_cross_entropy(y_train_arr, probabilities))
        errors = probabilities - y_train_arr
        dw = (X_train_arr.T @ errors) / n_samples
        db = np.mean(errors)
        weights -= lr * dw
        bias -= lr * db

    return weights, bias, loss_history

### TASK 7.4 — Evaluate scratch LR

1. Use learned weights and bias to compute probabilities.
2. Convert probabilities to class labels using threshold 0.5.
3. Report classification metrics.
4. Plot the loss curve.

In [ ]:
# TASK 7.4: Evaluate scratch LR
def calculate_classification_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1-score": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

# Use the same reduced split as scratch kNN so the scratch models are practical to train.
y_train_binary_knn = np.asarray([label_mapping[label] for label in y_train_knn], dtype=int)
y_test_binary_knn = np.asarray([label_mapping[label] for label in y_test_knn], dtype=int)
scratch_lr_weights, scratch_lr_bias, scratch_lr_loss = train_logistic_regression_scratch(
    X_train_knn, y_train_binary_knn, lr=0.1, epochs=300
)
scratch_lr_test_probabilities = sigmoid(X_test_knn @ scratch_lr_weights + scratch_lr_bias)
scratch_lr_test_predictions = (scratch_lr_test_probabilities >= 0.5).astype(int)
scratch_lr_metrics = calculate_classification_metrics(
    y_test_binary_knn, scratch_lr_test_predictions
)

print("Scratch Logistic Regression metrics:")
print(pd.Series(scratch_lr_metrics).round(4).to_string())
print("\nConfusion matrix (labels 0=normal, 1=malicious):")
print(confusion_matrix(y_test_binary_knn, scratch_lr_test_predictions, labels=[0, 1]))
print("\nClassification report:")
print(classification_report(
    y_test_binary_knn, scratch_lr_test_predictions,
    labels=[0, 1], target_names=["normal", "malicious"], zero_division=0, digits=4
))

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(scratch_lr_loss) + 1), scratch_lr_loss)
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy loss")
plt.title("Scratch Logistic Regression Training Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## SECTION 8 — scikit-learn Logistic Regression

### TASK 8.1 — Train and evaluate LogisticRegression

Use a pipeline with the same preprocessing.

Compare LR with kNN.

In [ ]:
# TASK 8.1: LogisticRegression pipeline
lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])
lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)
sklearn_lr_metrics = calculate_classification_metrics(y_test, y_pred_lr)

print("scikit-learn Logistic Regression metrics:")
print(pd.Series(sklearn_lr_metrics).round(4).to_string())
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_lr))
print("\nClassification report:")
print(classification_report(y_test, y_pred_lr, zero_division=0, digits=4))

## SECTION 9 — Final comparison and discussion

### TASK 9.1 — Create a comparison table

Compare:

1. scratch kNN
2. tuned scikit-learn kNN
3. scratch Logistic Regression
4. scikit-learn Logistic Regression

### Discussion questions

1. Which model gives the best F1-score?
2. Which model is more suitable for real-time IoT security detection?
3. Why is scaling important for kNN?
4. Why might LR underperform on nonlinear attack patterns?
5. What happens when k is too small or too large?

In [ ]:
# TASK 9.1: Final comparison table
# Compare every model on the same test rows used by scratch kNN.
comparison_n = len(y_test_knn)
comparison_y_test = y_test.iloc[:comparison_n]
best_scratch_knn_row = results_df.loc[results_df["F1-score"].idxmax()]
tuned_knn_comparison_predictions = best_knn.predict(X_test.iloc[:comparison_n])
sklearn_lr_comparison_predictions = lr_pipeline.predict(X_test.iloc[:comparison_n])

comparison_df = pd.DataFrame([
    {
        "Model": f"Scratch kNN (k={int(best_scratch_knn_row['k'])})",
        "Accuracy": best_scratch_knn_row["Accuracy"],
        "Precision": best_scratch_knn_row["Precision"],
        "Recall": best_scratch_knn_row["Recall"],
        "F1-score": best_scratch_knn_row["F1-score"],
    },
    {
        "Model": "Tuned scikit-learn kNN",
        **calculate_classification_metrics(comparison_y_test, tuned_knn_comparison_predictions),
    },
    {
        "Model": "Scratch Logistic Regression",
        **calculate_classification_metrics(y_test_binary_knn, scratch_lr_test_predictions),
    },
    {
        "Model": "scikit-learn Logistic Regression",
        **calculate_classification_metrics(comparison_y_test, sklearn_lr_comparison_predictions),
    },
]).sort_values("F1-score", ascending=False).reset_index(drop=True)

print(f"Model comparison on the same {comparison_n} test rows:")
print(comparison_df.round(4).to_string(index=False))
print(f"\nBest F1-score: {comparison_df.loc[0, 'Model']} "
      f"({comparison_df.loc[0, 'F1-score']:.4f})")